In [1]:
import pandas as pd

df = pd.read_csv("/srv/data1/general/immunopeptides_data/outputs/binding_score_function/1_processed/pdbs.csv")

df

,PDB code,Resolution,Release year,Binding data,Reference,Ligand name,protein_sequence,peptide_sequence
0,1uvt,2.50,1997,Ki=0.023uM,1uvt.pdf,(I48),IVEGQDAEVGLSPWQVMLFRKSPQELLCGASLISDRWVLTAAHCLL...,CGLRPLFEKKQVQDQTEKELFESYI
1,1yds,2.20,1997,Ki=1.2uM,1yds.pdf,(IQS),VKEFLAKAKEDFLKKWENPAQNTAHLDQFERIKTLGTGSFGRVMLV...,TTYADFIASGRTGRRNAIHD
2,1ydr,2.20,1997,Ki=3.0uM,1ydr.pdf,(IQP),VKEFLAKAKEDFLKKWENPAQNTAHLDQFERIKTLGTGSFGRVMLV...,TTYADFIASGRTGRRNAIHD
3,1ydt,2.30,1997,Ki=48nM,1ydt.pdf,(IQB),VKEFLAKAKEDFLKKWENPAQNTAHLDQFERIKTLGTGSFGRVMLV...,TTYADFIASGRTGRRNAIHD
4,1rtf,2.30,1997,Ki=910uM,1rtf.pdf,(BEN),IKGGLFADIASHPWQAAIFAKHRGERFLCGGILISSCWILSAAHCF...,TCGLRQYS
...,...,...,...,...,...,...,...,...
311,6idx,1.70,2019,Kd=3.36uM,6idx.pdf,(25-mer),SDIVKVAIEWPGANAQLLEIDQKRPLASIIKEVCDGWSLPNPEYYT...,RKSRYAELDFEKIMHTRKRHQDMFQ
312,6fbx,1.64,2019,Kd=343nM,6fbx.pdf,(26-mer),GPLSMSCWLREQTLLLAEDYISFCSGIQQTPPSESAEAMRYLAKEM...,LWAAKKYGQQLRRMSDEFDKGQ
313,6gvl,2.05,2019,Kd=44uM,6gvk.pdf,(30-mer),VPDTPTRLVFSALGPTSLRVSWQEPRCPLQGYSVEYQLLNGGELHR...,SNENLLLVHCGPTLINSCISFGS
314,6jjw,2.40,2019,Kd=8.2nM,6jjw.pdf,(32-mer),LPLPEGWEEARDFDGKVYYIDHRNRTTSWIDPRDRYTKPLTFADCI...,IIVPSYRPTPDYETVMRQMK


In [2]:
df["protein_sequence"].nunique()

264

In [3]:
df["peptide_sequence"].nunique()

248

In [23]:
binding_data = df["Binding data"].str.extract(
    r"([=<>]?)(\d+\.?\d*)([a-zA-Z]*)"
)
binding_data[1] = pd.to_numeric(binding_data[1], errors="coerce")
unit_conversion = {
    "fM": 1e-15,
    "pM": 1e-12,
    "nM": 1e-9,
    "uM": 1e-6,
    "mM": 1e-3,
    "M": 1,
}
df["affinity"] = binding_data[1] * binding_data[2].map(unit_conversion)

# get affinity spread for same peptide protein-pair
def get_affinity_spread(df):
    if len(df) == 1:
        return 0
    else:
        return df.max() - df.min()

df["affinity_spread"] = (
    df.groupby(["protein_sequence", "peptide_sequence"])["affinity"]
    .transform(get_affinity_spread)
)

df.sort_values(
    "affinity_spread",
    ascending=False,
    inplace=True,
)
df[["protein_sequence", "peptide_sequence", "affinity", "affinity_spread"]].head(10)

,protein_sequence,peptide_sequence,affinity,affinity_spread
9,IIGGEFTTIENQPWFAAIYRRHRGGSVTYVCGGSLMSPCWVISATH...,LKFQCGQKT,6.300000e-05,0.000063
6,IIGGEFTTIENQPWFAAIYRRHRGGSVTYVCGGSLMSPCWVISATH...,LKFQCGQKT,2.100000e-07,0.000063
109,VDRGLASANVDFAFSLYKQLVLKAPDKNVIFSPVSISTALAFLSLG...,RTIVRFNRPFLMIIVDHFTWSIFFMSKVTNPKQA,3.800000e-06,0.000038
110,VDRGLASANVDFAFSLYKQLVLKAPDKNVIFSPVSISTALAFLSLG...,RTIVRFNRPFLMIIVDHFTWSIFFMSKVTNPKQA,4.160000e-05,0.000038
41,SVKEFLAKAKEDFLKKWENPAQNTAHLDQFERIKTLGTGSFGRVML...,TTYADFIASGRTGRRNAIHD,6.300000e-09,0.000006
34,SVKEFLAKAKEDFLKKWENPAQNTAHLDQFERIKTLGTGSFGRVML...,TTYADFIASGRTGRRNAIHD,3.400000e-08,0.000006
16,SVKEFLAKAKEDFLKKWENPAQNTAHLDQFERIKTLGTGSFGRVML...,TTYADFIASGRTGRRNAIHD,5.700000e-06,0.000006
27,IIGGEFTTIENQPWFAAIYRRHRGGSVTYVCGGSLMSPCWVISATH...,LKFQCGQKT,3.800000e-07,0.000006
30,IIGGEFTTIENQPWFAAIYRRHRGGSVTYVCGGSLMSPCWVISATH...,LKFQCGQKT,6.000000e-06,0.000006
101,CPTHADSLNNLANIKREQGNIEEAVRLYRKALEVFPEFAAAHSNLA...,THETGTTNTATTAT,4.100000e-06,0.000004
